# Link Analysis

Analyse relationships between HDB resale property listings (January 2017–September 2020) by constructing a property listing network, examining its components and running centrality algorithms (**Degree Centrality** and **PageRank Centrality**).

## Imports

Import the required libraries.

In [1]:
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from IPython.display import display

## Data Cleaning and Feature Engineering

### Load January 2017 onwards data and remove duplicates

Read `resale-flat-prices-based-on-registration-date-from-jan-2017-onwards.csv`, parse `month` as a monthly period and drop duplicate rows.

In [2]:
# Read the January 2017 onwards CSV and sort chronologically.
df = pd.read_csv(
    "data/resale-flat-prices-based-on-registration-date-from-jan-2017-onwards.csv",
    dtype={"block": "string", "remaining_lease": "string"},
)
df["month"] = pd.to_datetime(df["month"], format="%Y-%m").dt.to_period("M")
df.sort_values("month", inplace=True, kind="mergesort", ignore_index=True)

original_count = len(df)
df.drop_duplicates(ignore_index=True, inplace=True)

print(f"Original rows: {original_count:,}")
print(f"Removed duplicate rows: {original_count - len(df):,}")
print(f"Cleaned rows: {len(df):,}")
df

Original rows: 80,374
Removed duplicate rows: 210
Cleaned rows: 80,164


/var/folders/8l/_7g518rx3js3_kngmmslndl40000gn/T/ipykernel_23618/3765301040.py:7: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.sort_values("month", inplace=True, kind="mergesort", ignore_index=True)
/var/folders/8l/_7g518rx3js3_kngmmslndl40000gn/T/ipykernel_23618/3765301040.py:10: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop_duplicates(ignore_index=True, inplace=True)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0
...,...,...,...,...,...,...,...,...,...,...,...
80159,2020-09,YISHUN,5 ROOM,716,YISHUN ST 71,07 TO 09,131.0,Improved,1987,66 years 03 months,440000.0
80160,2020-09,YISHUN,5 ROOM,760,YISHUN ST 72,07 TO 09,122.0,Improved,1987,65 years 06 months,458000.0
80161,2020-09,YISHUN,5 ROOM,835,YISHUN ST 81,04 TO 06,122.0,Improved,1987,66 years 04 months,490000.0
80162,2020-09,YISHUN,EXECUTIVE,791,YISHUN AVE 2,04 TO 06,146.0,Maisonette,1987,66 years 03 months,558000.0


### Engineer features for graph construction

Create the features needed to link listings: `storey_midpoint`, `flat_age_years` and `block_address`.

In [3]:
# Standardise text labels.
df["flat_type"] = df["flat_type"].replace({"MULTI GENERATION": "MULTI-GENERATION"})
df["flat_model"] = df["flat_model"].astype("string").str.strip().str.upper()

# Extract storey midpoint, flat age in years and block address (block + street_name).
df["transaction_year"] = df["month"].dt.year
df[["storey_min", "storey_max"]] = df["storey_range"].str.split(" TO ", expand=True).astype(int)
df["storey_midpoint"] = ((df["storey_min"] + df["storey_max"]) / 2).astype(int)
df["flat_age_years"] = df["transaction_year"] - df["lease_commence_date"]
df["block_address"] = (
    df["block"].astype("string").str.strip()
    + " "
    + df["street_name"].astype("string").str.strip().str.upper()
)
df[["month", "town", "block_address", "flat_type", "floor_area_sqm", "storey_midpoint", "flat_age_years", "resale_price"]]

,month,town,block_address,flat_type,floor_area_sqm,storey_midpoint,flat_age_years,resale_price
0,2017-01,ANG MO KIO,406 ANG MO KIO AVE 10,2 ROOM,44.0,11,38,232000.0
1,2017-01,ANG MO KIO,108 ANG MO KIO AVE 4,3 ROOM,67.0,2,39,250000.0
2,2017-01,ANG MO KIO,602 ANG MO KIO AVE 5,3 ROOM,67.0,2,37,262000.0
3,2017-01,ANG MO KIO,465 ANG MO KIO AVE 10,3 ROOM,68.0,5,37,265000.0
4,2017-01,ANG MO KIO,601 ANG MO KIO AVE 5,3 ROOM,67.0,2,37,265000.0
...,...,...,...,...,...,...,...,...
80159,2020-09,YISHUN,716 YISHUN ST 71,5 ROOM,131.0,8,33,440000.0
80160,2020-09,YISHUN,760 YISHUN ST 72,5 ROOM,122.0,8,33,458000.0
80161,2020-09,YISHUN,835 YISHUN ST 81,5 ROOM,122.0,5,33,490000.0
80162,2020-09,YISHUN,791 YISHUN AVE 2,EXECUTIVE,146.0,5,33,558000.0


## Link Analysis Task I: Graph construction and component analysis

### Configure and construct the property listing graph

Each of the **80,164 property listings** is a node. Connect each listing to its **$k=5$ closest comparable listings** by combining standardised physical distance (`floor_area_sqm`, `flat_age_years`, `storey_midpoint`) and a tiered location penalty. The $k=5$ candidates with the lowest combined distance score are selected as connected neighbours.

| Location relationship | Penalty | Total distance |
|---|---|---|
| **Same town, same street, same block** | `0.0` | `Physical distance + 0.0` |
| **Same town, same street, diff block** | `+0.2` | `Physical distance + 0.2` |
| **Same town, diff street, diff block** | `+0.5` | `Physical distance + 0.5` |
| **Diff town, diff street, diff block** | No link | Not connected |

- Listings match with the same block (0.0) or street (+0.2) first, and only link to another street in the same town (+0.5) when it offers a closer match in size, age and floor level.
- Because edges are undirected, every listing has at least 5 links, and can have more than 5 if other listings also pick it as a top-5 match.

In [4]:
# Configure neighbour count, tiered location penalties and physical similarity features.
K_NEIGHBORS = 5
DIFF_BLOCK_PENALTY = 0.2   # Same town, same street, different block
DIFF_STREET_PENALTY = 0.5   # Same town, different street, different block
SIMILARITY_FEATURES = [
    "floor_area_sqm",
    "flat_age_years",
    "storey_midpoint",
]
assert "resale_price" not in SIMILARITY_FEATURES, "Exclude target price from edge construction"

# Assign a unique sequential node ID (0 to 80,163) to each property listing.
df = df.reset_index(drop=True).copy()
df["listing_id"] = np.arange(len(df), dtype=int)

# Standardise physical features (mean 0, SD 1) across all 80,164 listings so distances share a common scale.
scaler = StandardScaler()
coords_all = scaler.fit_transform(df[SIMILARITY_FEATURES]).astype(np.float32)

edges_u, edges_v, edges_dist, edges_tier = [], [], [], []

# Process one town at a time since cross-town listings are not connected.
for town, town_group in df.groupby("town"):
    idx = town_group["listing_id"].to_numpy()
    coords = coords_all[idx]
    blocks = town_group["block_address"].to_numpy()
    streets = town_group["street_name"].to_numpy()
    n = len(idx)

    # Compute pairwise Euclidean distance on standardised (floor_area_sqm, flat_age_years, storey_midpoint).
    sq_norms = (coords ** 2).sum(axis=1)
    d2 = np.maximum(0.0, sq_norms[:, None] + sq_norms[None, :] - 2.0 * (coords @ coords.T))
    phys_dist = np.sqrt(d2, dtype=np.float32)

    # Identify which pairs share the exact same block vs same street (different block) vs different street.
    same_block = (blocks[:, None] == blocks[None, :])
    same_street = (streets[:, None] == streets[None, :])

    # Apply the tiered location penalty (0.0 for same block, +0.2 for same street, +0.5 for different street).
    penalty = np.where(
        same_block, 0.0, np.where(same_street, DIFF_BLOCK_PENALTY, DIFF_STREET_PENALTY)
    ).astype(np.float32)
    total_dist = phys_dist + penalty
    np.fill_diagonal(total_dist, np.inf)  # Prevent a listing from connecting to itself.

    # Select the K_NEIGHBORS=5 listings with the smallest total distance for each listing.
    k = min(K_NEIGHBORS, n - 1)
    nbr_local = np.argpartition(total_dist, kth=k - 1, axis=1)[:, :k]

    # Record the directed top-K pairs, their total distances and their location tier (0, 1 or 2).
    row_rep = np.repeat(np.arange(n), k)
    col_flat = nbr_local.ravel()
    d_flat = total_dist[row_rep, col_flat]
    t_flat = np.where(
        same_block[row_rep, col_flat], 0,
        np.where(same_street[row_rep, col_flat], 1, 2),
    )

    edges_u.append(idx[row_rep])
    edges_v.append(idx[col_flat])
    edges_dist.append(d_flat)
    edges_tier.append(t_flat)

# Convert directed top-K links into unique undirected edges (u < v) and compute similarity weights w = 1 / (1 + d).
u_all = np.concatenate(edges_u)
v_all = np.concatenate(edges_v)
d_all = np.concatenate(edges_dist)
t_all = np.concatenate(edges_tier)

edge_df = pd.DataFrame({
    "u": np.minimum(u_all, v_all),
    "v": np.maximum(u_all, v_all),
    "distance": d_all,
    "tier": t_all,
}).sort_values("distance").drop_duplicates(subset=["u", "v"], keep="first").reset_index(drop=True)
edge_df["weight"] = 1.0 / (1.0 + edge_df["distance"])

# Populate the NetworkX undirected graph with all 80,164 listing nodes and weighted edges.
G = nx.Graph()
G.add_nodes_from(df["listing_id"].to_numpy())
G.add_edges_from(
    zip(
        edge_df["u"].to_numpy(),
        edge_df["v"].to_numpy(),
        [{"weight": float(w), "distance": float(d), "tier": int(t)}
         for w, d, t in zip(edge_df["weight"], edge_df["distance"], edge_df["tier"])],
    )
)

print(f"Constructed listing graph with {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges.")

Constructed listing graph with 80,164 nodes and 254,042 edges.


### High-level graph topology summary

Summarise node and edge counts, location-tier shares, degree statistics, clustering coefficient and connected components.

<b>Observation</b>: Over 90% of links connect listings in the same street (61.5%) or block (28.6%), with the remaining 9.9% bridging across streets in the same town. Despite being sparse overall, the graph has a high clustering coefficient (0.43) and an average degree of 6.34, indicating that similar listings group into tight local clusters across 995 connected components.

In [5]:
# Count unweighted and weighted connections (degree) for every listing node.
degrees = np.array([deg for _, deg in G.degree()])
weighted_degrees = np.array([wdeg for _, wdeg in G.degree(weight="weight")])

# Count how many undirected edges belong to each location tier (0 = same block, 1 = same street, 2 = same town).
tier_counts = edge_df["tier"].value_counts().reindex([0, 1, 2], fill_value=0)
total_edges = G.number_of_edges()

# Extract all connected components and sort them from largest to smallest node count.
components = sorted(nx.connected_components(G), key=len, reverse=True)
component_sizes = [len(c) for c in components]

# Build a summary table of global network topology metrics.
topology_summary = pd.DataFrame([
    ("Total nodes (property listings)", f"{G.number_of_nodes():,}"),
    ("Unique HDB blocks (mean listings per block)", f"{df['block_address'].nunique():,} ({len(df) / df['block_address'].nunique():.1f})"),
    ("Total undirected edges", f"{total_edges:,}"),
    ("Tier 0: Same town, same street, same block", f"{tier_counts[0]:,} ({tier_counts[0] / total_edges * 100:.1f}%)"),
    ("Tier 1: Same town, same street, diff block", f"{tier_counts[1]:,} ({tier_counts[1] / total_edges * 100:.1f}%)"),
    ("Tier 2: Same town, diff street, diff block", f"{tier_counts[2]:,} ({tier_counts[2] / total_edges * 100:.1f}%)"),
    ("Graph density", f"{nx.density(G):.6f}"),
    ("Average degree (min / median / max)", f"{degrees.mean():.2f} ({degrees.min()} / {int(np.median(degrees))} / {degrees.max()})"),
    ("Average weighted degree", f"{weighted_degrees.mean():.2f}"),
    ("Average clustering coefficient", f"{nx.average_clustering(G):.4f}"),
    ("Number of connected components", f"{len(components):,}"),
    ("Largest / median / smallest component size (nodes)", f"{component_sizes[0]:,} / {int(np.median(component_sizes)):,} / {component_sizes[-1]:,}"),
], columns=["Graph metric", "Value"])
topology_summary

,Graph metric,Value
0,Total nodes (property listings),"80,164"
1,Unique HDB blocks (mean listings per block),"8,859 (9.0)"
2,Total undirected edges,"254,042"
3,"Tier 0: Same town, same street, same block","72,693 (28.6%)"
4,"Tier 1: Same town, same street, diff block","156,321 (61.5%)"
5,"Tier 2: Same town, diff street, diff block","25,028 (9.9%)"
6,Graph density,0.000079
7,Average degree (min / median / max),6.34 (5 / 6 / 18)
8,Average weighted degree,5.01
9,Average clustering coefficient,0.5778


### Analyse connected components

Profile the largest connected components across towns, streets, blocks, flat types and median resale prices.

In [6]:
# Assign each listing its connected component ID (1 = largest component, 988 = smallest).
node_to_comp = np.empty(len(df), dtype=int)
for comp_id, comp_nodes in enumerate(components, start=1):
    node_to_comp[list(comp_nodes)] = comp_id
df["component_id"] = node_to_comp

# Count undirected edges inside each connected component.
edge_comp_counts = pd.Series(node_to_comp[edge_df["u"].to_numpy()]).value_counts()

# Summarise each connected component by town, listing count, edge count, streets/blocks covered and physical/price profile.
comp_summary = (
    df.groupby("component_id", as_index=False)
    .agg(
        town=("town", "first"),
        listings=("listing_id", "count"),
        unique_streets=("street_name", "nunique"),
        unique_blocks=("block_address", "nunique"),
        dominant_flat_type=("flat_type", lambda s: s.value_counts().index[0]),
        mean_floor_area_sqm=("floor_area_sqm", lambda s: round(s.mean(), 1)),
        mean_flat_age_years=("flat_age_years", lambda s: round(s.mean(), 1)),
        median_resale_price=("resale_price", lambda s: round(s.median(), 0)),
    )
    .assign(edges=lambda d: d["component_id"].map(edge_comp_counts).fillna(0).astype(int))
    .loc[:, [
        "component_id", "town", "listings", "edges", "unique_streets",
        "unique_blocks", "dominant_flat_type", "mean_floor_area_sqm",
        "mean_flat_age_years", "median_resale_price",
    ]]
    .sort_values("listings", ascending=False)
    .reset_index(drop=True)
)

print(f"Total connected components: {len(comp_summary):,}")
print(f"Components with >= 100 listings: {(comp_summary['listings'] >= 100).sum():,} "
      f"(covering {comp_summary.loc[comp_summary['listings'] >= 100, 'listings'].sum() / len(df) * 100:.1f}% of all listings)")
display(comp_summary.head(20))
display(comp_summary.tail(20))

Total connected components: 995
Components with >= 100 listings: 59 (covering 78.7% of all listings)


,component_id,town,listings,edges,unique_streets,unique_blocks,dominant_flat_type,mean_floor_area_sqm,mean_flat_age_years,median_resale_price
0,1,WOODLANDS,4707,14733,37,606,4 ROOM,109.6,24.0,360000.0
1,2,TAMPINES,4454,14197,32,679,4 ROOM,107.8,29.6,445000.0
2,3,JURONG WEST,4405,14099,37,653,4 ROOM,106.6,22.7,394000.0
3,4,YISHUN,3695,12008,23,545,4 ROOM,95.7,29.2,327000.0
4,5,BEDOK,3277,10439,25,468,4 ROOM,95.6,34.1,394000.0
5,6,HOUGANG,3194,10198,24,498,4 ROOM,103.9,28.9,395000.0
6,7,BUKIT MERAH,2992,9408,43,336,4 ROOM,87.9,27.3,589500.0
7,8,ANG MO KIO,2864,9255,15,332,3 ROOM,87.2,33.7,365000.0
8,9,CHOA CHU KANG,2596,8218,21,436,4 ROOM,115.3,23.5,370000.0
9,10,BUKIT BATOK,2528,7978,21,332,4 ROOM,97.9,29.9,362000.0


,component_id,town,listings,edges,unique_streets,unique_blocks,dominant_flat_type,mean_floor_area_sqm,mean_flat_age_years,median_resale_price
975,940,WOODLANDS,6,15,1,5,4 ROOM,100.5,18.2,340000.0
976,948,PASIR RIS,6,15,1,5,5 ROOM,121.8,23.3,470000.0
977,947,BUKIT MERAH,6,15,1,4,3 ROOM,60.0,10.2,469000.0
978,946,CHOA CHU KANG,6,15,1,6,4 ROOM,101.3,19.3,319500.0
979,945,BEDOK,6,15,1,1,3 ROOM,67.0,39.7,290000.0
980,944,WOODLANDS,6,15,1,3,5 ROOM,112.0,4.5,420000.0
981,943,TAMPINES,6,15,1,4,5 ROOM,129.3,22.5,509000.0
982,942,TAMPINES,6,15,1,4,4 ROOM,100.2,21.7,403000.0
983,941,WOODLANDS,6,15,1,4,4 ROOM,103.3,17.7,344000.0
984,939,TAMPINES,6,15,1,5,4 ROOM,104.0,34.5,472500.0


### Visualise edge distribution across tiers and component edge sizes

Plot the distribution of links across the three location tiers (`Tier 0: Same block`, `Tier 1: Same street, diff block`, `Tier 2: Same town, diff street`) on the top row, and the number of edges for all connected components ranked from highest to lowest edge count on the bottom row.

**Observation:** Most links connect flats across neighbouring blocks on the same street (`61.5%`, Tier 1) rather than inside the same block (`28.6%`, Tier 0). Because individual blocks only record a small number of sales at the same flat size and floor level over 2017–2020 (~9 sales per block on average), listings often find closer matches in adjacent blocks built in the same batch along the same street, while the remaining `9.9%` (Tier 2) bridge across streets when a flat's size or height is less common locally. In the bottom chart, the steep drop-off shows that HDB comparables are unevenly distributed: a small number of large, uniform estates form wide, highly interconnected neighbourhood markets with thousands of links, while most of the 995 components are small, self-contained pockets, typically isolated blocks or distinct flat types that match only within their immediate building or street.

In [7]:
# Count edges in each location tier and sort all connected components from max edges to smallest.
tier_labels = [
    "Tier 0<br>Same block (0.0)",
    "Tier 1<br>Same street (+0.2)",
    "Tier 2<br>Same town (+0.5)",
]
tier_values = [int(tier_counts[0]), int(tier_counts[1]), int(tier_counts[2])]
tier_shares = [v / total_edges * 100 for v in tier_values]

comps_by_edges = comp_summary.sort_values("edges", ascending=False).reset_index(drop=True)
comps_by_edges["rank"] = np.arange(1, len(comps_by_edges) + 1)

comp_fig = make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.44, 0.56],
    vertical_spacing=0.18,
    subplot_titles=[
        "Distribution of links across location tiers",
        f"Edges per connected component (all {len(comps_by_edges):,} components, ranked max to min)",
    ],
)

# Top row: thinner bars with horizontal x-axis labels so they do not collide with the bottom subplot title.
comp_fig.add_trace(
    go.Bar(
        x=tier_labels,
        y=tier_values,
        width=0.35,
        marker_color=["#125183", "#1f77b4", "#6baed6"],
        text=[f"{v:,} ({s:.1f}%)" for v, s in zip(tier_values, tier_shares)],
        textposition="outside",
        hovertemplate="%{x}<br>Edges: %{y:,}<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Bottom row: all connected components ordered from maximum edges to smallest.
comp_fig.add_trace(
    go.Scatter(
        x=comps_by_edges["rank"],
        y=comps_by_edges["edges"],
        mode="lines+markers",
        fill="tozeroy",
        fillcolor="rgba(31,119,180,0.2)",
        line=dict(color="#1f77b4", width=2),
        marker=dict(
            size=4,
            color=comps_by_edges["median_resale_price"],
            colorscale="Blues",
            colorbar=dict(
                title="Median price (SGD)",
                tickformat=",.0f",
                x=1.02,
                y=0.23,
                len=0.45,
            ),
        ),
        customdata=comps_by_edges[[
            "component_id", "town", "listings", "unique_streets",
            "unique_blocks", "dominant_flat_type", "median_resale_price",
        ]],
        hovertemplate=(
            "Rank: #%{x} (Component #%{customdata[0]} — %{customdata[1]})<br>"
            "Edges: %{y:,} | Listings: %{customdata[2]:,}<br>"
            "Streets: %{customdata[3]} | Blocks: %{customdata[4]}<br>"
            "Dominant type: %{customdata[5]}<br>"
            "Median price: SGD %{customdata[6]:,.0f}<extra></extra>"
        ),
        showlegend=False,
    ),
    row=2,
    col=1,
)

comp_fig.update_xaxes(title_text="Location tier", tickangle=0, row=1, col=1)
comp_fig.update_yaxes(title_text="Number of edges", tickformat=",.0f", range=[0, max(tier_values) * 1.18], row=1, col=1)
comp_fig.update_xaxes(title_text="Connected component rank (1 = max edges, 995 = smallest)", tickformat=",.0f", row=2, col=1)
comp_fig.update_yaxes(title_text="Number of edges in component", tickformat=",.0f", row=2, col=1)

comp_fig.update_layout(
    title="Property listing graph — link distribution by tier and component edge counts",
    height=950,
    autosize=True,
    template="plotly_white",
)
comp_fig.show(renderer="plotly_mimetype", config={"responsive": True})

### Visualise representative connected component networks (`BISHAN`, `TAMPINES`, `BUKIT BATOK`)

Display node-link networks for the largest connected components in **Bishan** (1,164 listings across 10 streets), **Tampines** (4,454 listings across 32 streets) and **Bukit Batok** (2,528 listings across 21 streets) using a spring layout (`nx.spring_layout`, seed 42). Each subplot uses its own colour scale (`resale_price`) so internal price tiers within each town are clearly visible. Solid grey lines show same-block links (`0.0`) and dotted blue lines show cross-block / cross-street links (`+0.2 / +0.5`).

**Observation:** Across both compact central towns (Bishan) and large multi-neighbourhood towns (Tampines and Bukit Batok), listings naturally separate into distinct price branches even though `resale_price` was never used to build the network—higher-priced larger or higher-floor flats (green/yellow) cluster on one side, while smaller or older flats (purple/blue) cluster on another. Solid grey links anchor each block into tight local groups, while dotted blue bridges link comparable flats across streets, illustrating how price benchmarks carry from one block or street pocket to similar flats elsewhere in the town.

In [8]:
# Select one representative component (~180–450 listings) each from BISHAN, TAMPINES and BUKIT BATOK.
sample_towns = ["BISHAN", "TAMPINES", "BUKIT BATOK"]
selected_comps = []
for town_name in sample_towns:
    town_comps = comp_summary.loc[
        (comp_summary["town"] == town_name) & (comp_summary["listings"].between(180, 450))
    ]
    if town_comps.empty:
        town_comps = comp_summary.loc[comp_summary["town"] == town_name]
    comp_row = town_comps.iloc[0]
    selected_comps.append((town_name, int(comp_row["component_id"]), int(comp_row["listings"]), int(comp_row["edges"])))

subplot_titles = [
    f"{t} — Component #{cid} ({n_nodes} listings, {n_edges} edges)"
    for t, cid, n_nodes, n_edges in selected_comps
]

net_fig = make_subplots(
    rows=3,
    cols=1,
    vertical_spacing=0.07,
    subplot_titles=subplot_titles,
)

# Position a separate price colorbar alongside each of the three stacked subplots.
colorbar_y_positions = [0.85, 0.50, 0.15]

for row_idx, (town_name, comp_id, _, _) in enumerate(selected_comps, start=1):
    sample_df = df.loc[df["component_id"] == comp_id].copy()
    sample_nodes = sample_df["listing_id"].tolist()
    subG = G.subgraph(sample_nodes)
    pos = nx.spring_layout(subG, weight="weight", seed=42, k=0.25, iterations=70)

    # Separate edge coordinates into same-block links (Tier 0) and cross-block / cross-street links (Tier 1 & 2).
    tier0_x, tier0_y = [], []
    tier12_x, tier12_y = [], []
    for u, v, edata in subG.edges(data=True):
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        if edata.get("tier", 0) == 0:
            tier0_x.extend([x0, x1, None])
            tier0_y.extend([y0, y1, None])
        else:
            tier12_x.extend([x0, x1, None])
            tier12_y.extend([y0, y1, None])

    node_x = [pos[n][0] for n in sample_nodes]
    node_y = [pos[n][1] for n in sample_nodes]
    node_prices = sample_df["resale_price"].to_numpy()
    node_degrees = [subG.degree(n) for n in sample_nodes]
    node_custom = sample_df[[
        "listing_id", "month", "block_address", "flat_type",
        "storey_range", "floor_area_sqm", "flat_age_years",
    ]].assign(month=lambda d: d["month"].astype(str), degree=node_degrees).to_numpy()

    net_fig.add_trace(
        go.Scatter(
            x=tier0_x, y=tier0_y, mode="lines",
            line=dict(width=0.9, color="rgba(100,100,100,0.45)"),
            hoverinfo="skip", name="Same-block links (0.0 penalty)",
            legendgroup="tier0", showlegend=(row_idx == 1),
        ),
        row=row_idx, col=1,
    )
    net_fig.add_trace(
        go.Scatter(
            x=tier12_x, y=tier12_y, mode="lines",
            line=dict(width=1.0, color="rgba(31,119,180,0.55)", dash="dot"),
            hoverinfo="skip", name="Cross-block / cross-street links (+0.2 / +0.5 penalty)",
            legendgroup="tier12", showlegend=(row_idx == 1),
        ),
        row=row_idx, col=1,
    )
    net_fig.add_trace(
        go.Scatter(
            x=node_x, y=node_y, mode="markers",
            name="Property listings",
            legendgroup="nodes", showlegend=(row_idx == 1),
            marker=dict(
                size=[5 + deg * 0.8 for deg in node_degrees],
                color=node_prices,
                colorscale="Viridis",
                showscale=True,
                colorbar=dict(
                    title=f"{town_name}<br>Price (SGD)",
                    tickformat=",.0f",
                    y=colorbar_y_positions[row_idx - 1],
                    len=0.28,
                ),
                line=dict(width=0.6, color="white"),
            ),
            customdata=node_custom,
            hovertemplate=(
                f"Town: {town_name}<br>"
                "Listing ID: %{customdata[0]} (%{customdata[1]})<br>"
                "Block: %{customdata[2]}<br>Type: %{customdata[3]} (Storey %{customdata[4]})<br>"
                "Area: %{customdata[5]} sqm | Age: %{customdata[6]} yrs<br>"
                "Resale price: SGD %{marker.color:,.0f}<br>Degree: %{customdata[7]}<extra></extra>"
            ),
        ),
        row=row_idx, col=1,
    )
    net_fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, title="", row=row_idx, col=1)
    net_fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, title="", row=row_idx, col=1)

net_fig.update_layout(
    title=dict(text="Property listing subgraphs — BISHAN, TAMPINES and BUKIT BATOK", y=0.98),
    height=1900,
    autosize=True,
    margin=dict(t=140, b=40),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=1.035, yanchor="bottom"),
    template="plotly_white",
)
net_fig.show(renderer="plotly_mimetype", config={"responsive": True})

## Link Analysis Task II: Graph algorithms and interpretation

2 centrality algorithms were run on the listing network to measure how often a listing is chosen as a comparable (**Degree Centrality**) and how central it is within its cluster (**PageRank Centrality**).

| Algorithm | How it works | Interpretation |
|---|---|---|
| **Degree Centrality** | Counts a node's connections divided by $(N - 1)$. Since each listing picks $k=5$ neighbours, any degree above 5 shows how many other listings picked that flat as a top-5 match. | High-degree listings are standard, typical flats in active blocks that many nearby sales match against. Degree-5 listings were not picked by any other listing and tend to be less typical floors or sizes. |
| **PageRank Centrality** | Ranks nodes using a weighted random walk: a listing scores higher when it is connected to many neighbours that are also well-connected and have higher similarity weights. | High-PageRank listings sit at the core of dense block or street clusters and match their neighbours' prices closely; low-PageRank listings sit on the edges of a cluster. |

### Algorithm 1: Degree centrality

**How it works:** Degree counts how many edges are connected to a listing, and Degree Centrality normalises that count by dividing by the maximum possible connections ($N - 1 = 80,163$). Because every listing picks its $k=5$ closest neighbours in an undirected graph, each listing starts with 5 outgoing links and gains incoming links (`incoming_selections = degree - 5`) whenever other nearby listings also select it as one of their top-5 matches. In this property network, a high degree indicates a "popular comparable" flat whose size, age, and floor level match many surrounding transactions in the same block or street, making it a common reference point for buyers and valuers.

**Why the output looks like this:** Degree centrality values appear as small decimals (`~0.00006` to `0.00022`) because degrees of `5` to `18` are divided by `80,163` total listings. Listings with the highest degrees (`14` to `18`) are low-to-mid-floor flats in active estates such as Punggol, Sengkang, Tampines, and Choa Chu Kang, where 9 to 13 neighbouring listings picked them as a top-5 comparable. By contrast, listings at the minimum degree (`degree = 5`, `incoming_selections = 0`) were not picked back by any other listing; these tend to be less typical units with rare flat models, extreme floor levels, or low-volume blocks, which explains why their prices diverge more from their connected neighbours.

In [9]:
# Compute Degree Centrality and raw node degree for every listing.
deg_centrality = nx.degree_centrality(G)
df["degree"] = degrees
df["degree_centrality"] = df["listing_id"].map(deg_centrality)
df["incoming_selections"] = df["degree"] - K_NEIGHBORS  # How many other listings picked this flat

# Compute the mean resale price of each listing's 1-hop graph neighbours.
prices_arr = df["resale_price"].to_numpy()
df["neighbor_mean_price"] = np.array([
    prices_arr[list(G.neighbors(n))].mean() for n in range(len(df))
])
df["abs_diff_from_neighbors"] = (df["resale_price"] - df["neighbor_mean_price"]).abs()

deg_cols = [
    "listing_id", "month", "town", "block_address", "flat_type",
    "storey_range", "floor_area_sqm", "flat_age_years", "resale_price",
    "degree", "incoming_selections", "degree_centrality",
]

print("Top 10 highest-degree listings (Most frequently selected comparables):")
display(df.sort_values(["degree", "resale_price"], ascending=[False, False])[deg_cols].head(10))

print("\nSample of lowest-degree listings (Degree = 5, selected by 0 other listings):")
display(df.loc[df["degree"] == K_NEIGHBORS].sort_values("abs_diff_from_neighbors", ascending=False)[deg_cols].head(10))

Top 10 highest-degree listings (Most frequently selected comparables):


,listing_id,month,town,block_address,flat_type,storey_range,floor_area_sqm,flat_age_years,resale_price,degree,incoming_selections,degree_centrality
71799,71799,2020-06,PUNGGOL,308C PUNGGOL WALK,4 ROOM,04 TO 06,92.0,4,515000.0,18,13,0.000225
73399,73399,2020-07,CHOA CHU KANG,487C CHOA CHU KANG AVE 5,4 ROOM,04 TO 06,93.0,4,403000.0,17,12,0.000212
72119,72119,2020-06,SENGKANG,416B FERNVALE LINK,5 ROOM,10 TO 12,113.0,4,585000.0,15,10,0.000187
13864,13864,2017-09,CLEMENTI,116 CLEMENTI ST 13,EXECUTIVE,10 TO 12,149.0,33,888000.0,14,9,0.000175
74817,74817,2020-07,TAMPINES,858 TAMPINES AVE 5,EXECUTIVE,01 TO 03,146.0,32,740000.0,14,9,0.000175
25643,25643,2018-04,PUNGGOL,289C PUNGGOL PL,5 ROOM,13 TO 15,112.0,5,622000.0,14,9,0.000175
63493,63493,2019-12,SENGKANG,440C FERNVALE LINK,5 ROOM,07 TO 09,113.0,4,550000.0,14,9,0.000175
76118,76118,2020-08,HOUGANG,443 HOUGANG AVE 8,5 ROOM,10 TO 12,121.0,30,550000.0,14,9,0.000175
12189,12189,2017-08,JURONG EAST,234 JURONG EAST ST 21,5 ROOM,10 TO 12,121.0,35,520000.0,14,9,0.000175
55568,55568,2019-08,SENGKANG,211C COMPASSVALE LANE,4 ROOM,07 TO 09,93.0,6,436000.0,14,9,0.000175



Sample of lowest-degree listings (Degree = 5, selected by 0 other listings):


,listing_id,month,town,block_address,flat_type,storey_range,floor_area_sqm,flat_age_years,resale_price,degree,incoming_selections,degree_centrality
75648,75648,2020-08,BUKIT MERAH,43 MOH GUAN TER,4 ROOM,04 TO 06,131.0,47,1088000.0,5,0,0.000062
61387,61387,2019-11,QUEENSTOWN,82 C'WEALTH CL,4 ROOM,01 TO 03,92.0,52,400000.0,5,0,0.000062
17142,17142,2017-11,BUKIT MERAH,43 MOH GUAN TER,3 ROOM,04 TO 06,77.0,44,630000.0,5,0,0.000062
56485,56485,2019-09,BISHAN,22 SIN MING RD,5 ROOM,10 TO 12,136.0,46,473000.0,5,0,0.000062
35596,35596,2018-09,KALLANG/WHAMPOA,41 JLN BAHAGIA,3 ROOM,01 TO 03,237.0,46,1185000.0,5,0,0.000062
38206,38206,2018-10,TAMPINES,460 TAMPINES ST 42,MULTI-GENERATION,10 TO 12,132.0,31,778000.0,5,0,0.000062
34849,34849,2018-09,BISHAN,22 SIN MING RD,5 ROOM,04 TO 06,136.0,45,515000.0,5,0,0.000062
64744,64744,2020-01,HOUGANG,475A UPP SERANGOON CRES,4 ROOM,10 TO 12,92.0,6,700000.0,5,0,0.000062
54661,54661,2019-08,BUKIT MERAH,9B BOON TIONG RD,4 ROOM,40 TO 42,92.0,3,1050000.0,5,0,0.000062
46793,46793,2019-04,BUKIT MERAH,87 ZION RD,5 ROOM,16 TO 18,117.0,45,825000.0,5,0,0.000062


### Visualise Degree Centrality distribution and neighbour price gap

Plot the number of listings by node degree (top) and the median price difference from connected neighbours across degree levels (bottom).

**Observation:** Most listings (`64.3%`) have a degree above 5, while `35.7%` remain at the minimum degree of 5 and very few exceed degree 9. The median price gap from neighbours stays around SGD 15,100–16,200 across degrees 5 to 11 before dropping to SGD 13,286 at degree 12+, showing that the most connected listings match neighbour prices most closely.

In [10]:
# Summarise listing counts and median neighbour price gap across node degree levels (grouping 12+ together for sample size).
df["degree_group"] = df["degree"].clip(upper=12).astype(str)
df.loc[df["degree"] >= 12, "degree_group"] = "12+"
deg_order = ["5", "6", "7", "8", "9", "10", "11", "12+"]

deg_stats = (
    df.groupby("degree_group")
    .agg(
        listings=("listing_id", "count"),
        median_abs_gap=("abs_diff_from_neighbors", "median"),
    )
    .reindex(deg_order)
    .reset_index()
)

deg_fig = make_subplots(
    rows=2,
    cols=1,
    vertical_spacing=0.18,
    subplot_titles=[
        "Listing count by node degree (all 80,164 listings)",
        "Median price difference from 1-hop neighbours by node degree",
    ],
)

# Top row: bar chart of listing counts by node degree.
deg_fig.add_trace(
    go.Bar(
        x=deg_stats["degree_group"],
        y=deg_stats["listings"],
        width=0.45,
        marker_color="#1f77b4",
        text=[f"{v:,}" for v in deg_stats["listings"]],
        textposition="outside",
        hovertemplate="Node degree: %{x}<br>Listings: %{y:,}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=1,
)

# Bottom row: median absolute price difference between a listing and its 1-hop neighbours across degrees.
deg_fig.add_trace(
    go.Scatter(
        x=deg_stats["degree_group"],
        y=deg_stats["median_abs_gap"],
        mode="lines+markers+text",
        line=dict(color="#125183", width=3),
        marker=dict(size=8, color="#125183"),
        text=[f"SGD {v:,.0f}" for v in deg_stats["median_abs_gap"]],
        textposition="top center",
        hovertemplate="Node degree: %{x}<br>Median gap from neighbours: SGD %{y:,.0f}<extra></extra>",
        showlegend=False,
    ),
    row=2, col=1,
)

deg_fig.update_xaxes(title_text="Node degree (5 = 0 incoming selections)", type="category", row=1, col=1)
deg_fig.update_yaxes(title_text="Number of property listings", tickformat=",.0f", range=[0, deg_stats["listings"].max() * 1.18], row=1, col=1)
deg_fig.update_xaxes(title_text="Node degree", type="category", row=2, col=1)
deg_fig.update_yaxes(
    title_text="Median |price − neighbour mean price| (SGD)",
    tickformat=",.0f",
    range=[deg_stats["median_abs_gap"].min() * 0.90, deg_stats["median_abs_gap"].max() * 1.08],
    row=2, col=1,
)

deg_fig.update_layout(
    title="Degree centrality analysis — degree distribution and neighbour price stability",
    height=900,
    autosize=True,
    template="plotly_white",
)
deg_fig.show(renderer="plotly_mimetype", config={"responsive": True})

### Algorithm 2: PageRank centrality

**How it works:** Imagine a buyer browsing comparable flats by clicking from one connected listing to the next, more often following stronger links (high similarity in size, age, floor level, and location) and occasionally jumping to a different listing. PageRank measures how often a buyer would land on each listing during this browsing process. Unlike Degree Centrality, which treats all links equally and just counts how many a flat has, PageRank gives a listing a higher score when it has strong similarity links to other well-connected listings. In the HDB resale context, this highlights core "benchmark" flats sitting in the middle of tightly knit groups of very similar flats, rather than flats with only loose or peripheral connections.

**Why the output looks like this:** PageRank values sit around `~0.000007` to `0.000028` because PageRank forms a probability distribution across all `80,164` listings that sums to `1.0`, giving an average listing a score of $1 / 80,164 \approx 0.0000125$. The highest-PageRank listings sit at the centre of tight, high-similarity block and street clusters where similar flats mutually link to one another, whereas the lowest-PageRank listings sit on the outer edges of a cluster with `degree = 5` and weaker similarity weights to their neighbours.

In [11]:
# Compute weighted PageRank centrality and node degree for all 80,164 listings.
pagerank_scores = nx.pagerank(G, weight="weight")
df["pagerank"] = df["listing_id"].map(pagerank_scores)
df["degree"] = degrees

# Compute the mean resale price of each listing's 1-hop graph neighbours to measure price alignment.
prices_arr = df["resale_price"].to_numpy()
neighbor_mean_prices = np.array([
    prices_arr[list(G.neighbors(n))].mean() for n in range(len(df))
])
df["neighbor_mean_price"] = neighbor_mean_prices
df["abs_diff_from_neighbors"] = (df["resale_price"] - df["neighbor_mean_price"]).abs()

cols_to_show = [
    "listing_id", "month", "town", "block_address", "flat_type", "flat_model",
    "storey_range", "floor_area_sqm", "flat_age_years", "resale_price",
    "degree", "pagerank",
]

print("Top 10 highest-PageRank listings (Benchmark / Anchor comparables):")
display(df.sort_values("pagerank", ascending=False)[cols_to_show].head(10))

print("\nBottom 10 lowest-PageRank listings (Peripheral / Atypical listings):")
display(df.sort_values("pagerank", ascending=True)[cols_to_show].head(10))

Top 10 highest-PageRank listings (Benchmark / Anchor comparables):


,listing_id,month,town,block_address,flat_type,flat_model,storey_range,floor_area_sqm,flat_age_years,resale_price,degree,pagerank
71799,71799,2020-06,PUNGGOL,308C PUNGGOL WALK,4 ROOM,PREMIUM APARTMENT,04 TO 06,92.0,4,515000.0,18,0.000028
43220,43220,2019-01,WOODLANDS,579 WOODLANDS DR 16,4 ROOM,MODEL A,10 TO 12,90.0,18,320000.0,14,0.000024
54639,54639,2019-08,BUKIT MERAH,2 JLN BT MERAH,3 ROOM,IMPROVED,07 TO 09,63.0,44,274000.0,13,0.000024
73399,73399,2020-07,CHOA CHU KANG,487C CHOA CHU KANG AVE 5,4 ROOM,MODEL A,04 TO 06,93.0,4,403000.0,17,0.000023
48526,48526,2019-05,BUKIT BATOK,239 BT BATOK EAST AVE 5,3 ROOM,NEW GENERATION,01 TO 03,67.0,34,225000.0,13,0.000023
29599,29599,2018-06,TAMPINES,253 TAMPINES ST 21,3 ROOM,MODEL A,10 TO 12,74.0,34,368000.0,14,0.000023
61255,61255,2019-11,PUNGGOL,624C PUNGGOL CTRL,2 ROOM,MODEL A,16 TO 18,46.0,5,273000.0,14,0.000023
46025,46025,2019-03,TAMPINES,234 TAMPINES ST 21,4 ROOM,MODEL A,04 TO 06,104.0,35,465000.0,13,0.000023
44960,44960,2019-03,BUKIT BATOK,505 BT BATOK ST 52,3 ROOM,IMPROVED,07 TO 09,60.0,31,230000.0,13,0.000023
55568,55568,2019-08,SENGKANG,211C COMPASSVALE LANE,4 ROOM,MODEL A,07 TO 09,93.0,6,436000.0,14,0.000023



Bottom 10 lowest-PageRank listings (Peripheral / Atypical listings):


,listing_id,month,town,block_address,flat_type,flat_model,storey_range,floor_area_sqm,flat_age_years,resale_price,degree,pagerank
62401,62401,2019-12,BISHAN,25 SIN MING RD,4 ROOM,IMPROVED,10 TO 12,88.0,45,380000.0,5,0.000007
78882,78882,2020-09,PASIR RIS,530A PASIR RIS DR 1,4 ROOM,DBSS,10 TO 12,86.0,5,625000.0,5,0.000007
71320,71320,2020-06,HOUGANG,23 HOUGANG AVE 3,EXECUTIVE,ADJOINED FLAT,10 TO 12,174.0,42,628000.0,5,0.000007
68472,68472,2020-03,JURONG WEST,210 BOON LAY PL,5 ROOM,ADJOINED FLAT,07 TO 09,139.0,43,400000.0,5,0.000007
26065,26065,2018-04,TOA PAYOH,168 LOR 1 TOA PAYOH,5 ROOM,ADJOINED FLAT,10 TO 12,150.0,46,735000.0,5,0.000007
78878,78878,2020-09,PASIR RIS,530B PASIR RIS DR 1,3 ROOM,DBSS,04 TO 06,65.0,5,445000.0,5,0.000007
76706,76706,2020-08,QUEENSTOWN,87 DAWSON RD,3 ROOM,PREMIUM APARTMENT,46 TO 48,65.0,4,698000.0,5,0.000007
26309,26309,2018-05,ANG MO KIO,226B ANG MO KIO AVE 1,3 ROOM,IMPROVED,01 TO 03,66.0,24,242000.0,5,0.000007
71679,71679,2020-06,PASIR RIS,530C PASIR RIS DR 1,5 ROOM,DBSS,04 TO 06,105.0,5,790000.0,5,0.000007
39446,39446,2018-11,JURONG WEST,337C TAH CHING RD,5 ROOM,IMPROVED,13 TO 15,112.0,8,530000.0,5,0.000007


### Visualise PageRank centrality and neighbour price correlation

Plot the median price difference from 1-hop neighbours across PageRank quintiles (top) and actual listing price versus 1-hop neighbour mean price (bottom, 10,000 sampled listings).

**Observation:** Listings in the top PageRank quintiles (Q3–Q5) have a smaller median price gap from their neighbours (~SGD 15,000–15,300) than the lowest PageRank quintile (Q1, ~SGD 17,200). Across the graph, a listing's price correlates strongly with the average price of its connected neighbours ($r = 0.980$), showing that the physical and location links capture real price relationships.

In [12]:
# Group listings into 5 PageRank quintiles (Q1 = lowest PageRank, Q5 = highest PageRank).
df["pagerank_quintile"] = pd.qcut(
    df["pagerank"],
    q=5,
    labels=["Q1 (Lowest)", "Q2", "Q3", "Q4", "Q5 (Highest)"],
)
pr_quintile_stats = (
    df.groupby("pagerank_quintile", observed=False)
    .agg(
        median_abs_gap=("abs_diff_from_neighbors", "median"),
        mean_degree=("degree", "mean"),
        listings=("listing_id", "count"),
    )
    .reset_index()
)

neighbor_corr = df["resale_price"].corr(df["neighbor_mean_price"])
pr_sample = df.sample(n=10_000, random_state=42)

pr_fig = make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.42, 0.58],
    vertical_spacing=0.18,
    subplot_titles=[
        "Price deviation from neighbours by PageRank quintile",
        f"Listing price vs 1-hop neighbour mean price (r = {neighbor_corr:.3f})",
    ],
)

# Top row: median absolute price difference from 1-hop neighbours across PageRank quintiles.
pr_fig.add_trace(
    go.Bar(
        x=pr_quintile_stats["pagerank_quintile"],
        y=pr_quintile_stats["median_abs_gap"],
        width=0.40,
        marker_color=["#6baed6", "#4292c6", "#2171b5", "#08519c", "#08306b"],
        text=[f"SGD {v:,.0f} (deg {d:.1f})" for v, d in zip(pr_quintile_stats["median_abs_gap"], pr_quintile_stats["mean_degree"])],
        textposition="outside",
        hovertemplate="PageRank quintile: %{x}<br>Median gap from neighbours: SGD %{y:,.0f}<extra></extra>",
        showlegend=False,
    ),
    row=1, col=1,
)

# Bottom row: actual price vs 1-hop neighbour mean price coloured by PageRank score.
pr_fig.add_trace(
    go.Scattergl(
        x=pr_sample["neighbor_mean_price"],
        y=pr_sample["resale_price"],
        mode="markers",
        marker=dict(
            size=4,
            color=pr_sample["pagerank"] * 1e5,
            colorscale="Viridis",
            opacity=0.45,
            showscale=True,
            colorbar=dict(title="PageRank (×10⁻⁵)", y=0.23, len=0.48),
        ),
        customdata=pr_sample[["listing_id", "town", "block_address", "flat_type", "degree"]],
        hovertemplate=(
            "Listing ID: %{customdata[0]} (%{customdata[1]})<br>"
            "Block: %{customdata[2]} (%{customdata[3]})<br>"
            "Actual price: SGD %{y:,.0f}<br>Neighbour mean: SGD %{x:,.0f}<br>"
            "Degree: %{customdata[4]}<extra></extra>"
        ),
        showlegend=False,
    ),
    row=2, col=1,
)

# Add y = x reference line.
min_p, max_p = float(df["resale_price"].min()), float(df["resale_price"].max())
pr_fig.add_trace(
    go.Scatter(
        x=[min_p, max_p], y=[min_p, max_p],
        mode="lines", line=dict(color="#d62728", width=1.5, dash="dash"),
        hoverinfo="skip", showlegend=False,
    ),
    row=2, col=1,
)

pr_fig.update_xaxes(title_text="PageRank centrality quintile", tickangle=0, row=1, col=1)
pr_fig.update_yaxes(
    title_text="Median |price − neighbour mean price| (SGD)",
    tickformat=",.0f",
    range=[0, pr_quintile_stats["median_abs_gap"].max() * 1.22],
    row=1, col=1,
)
pr_fig.update_xaxes(title_text="1-hop graph neighbour mean resale price (SGD)", tickformat=",.0f", row=2, col=1)
pr_fig.update_yaxes(title_text="Actual listing resale price (SGD)", tickformat=",.0f", row=2, col=1)

pr_fig.update_layout(
    title="PageRank centrality analysis — benchmark stability and neighbour price correlation",
    height=950,
    autosize=True,
    template="plotly_white",
)
pr_fig.show(renderer="plotly_mimetype", config={"responsive": True})